# Train ADAS xe thật — YOLOv8n-seg (bản thử nghiệm)
Upload ADAS_REAL_MERGED.zip vào thư mục gốc My Drive, chọn Runtime > Change runtime type > T4 GPU rồi chạy các cell theo thứ tự.
Dataset đã augmentation. Giữ nguyên split; không chia ngẫu nhiên lại. Crosswalk có cả nhãn từng thanh và nhãn cả vùng. Test chỉ có 4 đối tượng crosswalk: không dùng metrics này để khẳng định tính năng giảm tốc đã đáng tin cậy.
Tham khảo API: https://docs.ultralytics.com/tasks/segment/ và https://docs.ultralytics.com/modes/train/ .

In [ ]:
%pip install -q ultralytics==8.4.202


In [ ]:
from google.colab import drive
from pathlib import Path
import torch, zipfile, yaml, json, hashlib, subprocess
from datetime import datetime
from ultralytics import YOLO
import ultralytics
drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Hãy bật T4 GPU trong Runtime.'
print(torch.cuda.get_device_name(0), 'ultralytics', ultralytics.__version__)
ZIP = Path('/content/drive/MyDrive/ADAS_REAL_MERGED.zip')
assert ZIP.is_file(), f'Chưa thấy {ZIP}'
ROOT = Path('/content/adas_real_merged')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(ROOT)
cfg = yaml.safe_load((ROOT/'data.yaml').read_text())
assert cfg['names'] == {0:'lane_marking', 1:'crosswalk'}
cfg['path'] = str(ROOT)
(ROOT/'data.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False))
print((ROOT/'merge_report.json').read_text())
manifest = json.loads((ROOT/'manifest.json').read_text())
for key in ('family','digest','block'):
    groups = {}
    for item in manifest:
        groups.setdefault(str(item[key]), set()).add(item['split'])
    assert all(len(s)==1 for s in groups.values()), key
for split in ('train','valid','test'):
    images = list((ROOT/split/'images').glob('*.jpg'))
    assert images
    for im in images:
        label=ROOT/split/'labels'/(im.stem+'.txt')
        for row in label.read_text().splitlines():
            a=row.split()
            assert len(a)>=7 and len(a)%2==1 and int(a[0]) in (0,1)
            assert all(0<=float(v)<=1 for v in a[1:])
    print(split, len(images))


In [ ]:
# Lưu checkpoint trực tiếp lên Drive để hạn chế mất kết quả khi Colab ngắt.
RUN = Path('/content/drive/MyDrive/ADAS_training') / ('real_v8n_seg_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
RUN.mkdir(parents=True, exist_ok=False)
(RUN/'environment.txt').write_text(subprocess.check_output(['pip','freeze'], text=True))
(RUN/'dataset_sha256.txt').write_text(hashlib.sha256(ZIP.read_bytes()).hexdigest())
(RUN/'merge_report.json').write_text((ROOT/'merge_report.json').read_text())
model = YOLO('yolov8n-seg.pt')
model.train(data=str(ROOT/'data.yaml'), epochs=100, patience=20,
    imgsz=640, batch=8, device=0, workers=2, seed=42,
    project=str(RUN.parent), name=RUN.name, exist_ok=True,
    save=True, save_period=10, plots=True, cache=False,
    mosaic=0.0, mixup=0.0, copy_paste=0.0,
    degrees=0.0, translate=0.0, scale=0.0, shear=0.0, perspective=0.0,
    flipud=0.0, fliplr=0.0, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0)
BEST = Path(model.trainer.save_dir)/'weights/best.pt'
assert BEST.is_file()
print('Model:', BEST)


In [ ]:
# Chỉ đánh giá test sau khi hoàn tất chọn model bằng validation.
from IPython.display import display, Image
import cv2
best = YOLO(str(BEST))
metrics = best.val(data=str(ROOT/'data.yaml'), split='test', imgsz=640, batch=8, device=0,
                   project=str(RUN), name='test_evaluation')
print(metrics.results_dict)
(RUN/'test_metrics.json').write_text(json.dumps(metrics.results_dict, indent=2))
samples = sorted((ROOT/'test/images').glob('*.jpg'))
for im in samples[::max(1,len(samples)//6)][:6]:
    result=best.predict(str(im), imgsz=640, conf=0.25, device=0, verbose=False)[0]
    display(Image(data=cv2.imencode('.jpg',result.plot())[1].tobytes()))
from google.colab import files
files.download(str(BEST))


## Nếu Colab ngắt giữa chừng
Chạy lại cell cài đặt và mount/giải nén. Mở đường dẫn run trong Drive, dùng checkpoint last.pt để resume. Không chạy lại cell tạo run mới nếu muốn nối tiếp lượt cũ.
```python
from ultralytics import YOLO
YOLO('/content/drive/MyDrive/ADAS_training/TEN_RUN/weights/last.pt').train(resume=True)
```
Nếu hết bộ nhớ GPU ở lượt train mới, giảm batch từ 8 xuống 4. Model tải về đặt tại models/best.pt của dự án xe thật. Chưa train hoặc đo tốc độ Pi 5 trong notebook này trước khi bạn chạy.